# PEAK M-ATH — DNS Beaconing Detection

Detect C2 beaconing in DNS traffic by identifying hosts with near-constant intervals between outbound DNS queries to the same domain, following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**Ref:** M02 — Beaconing behavior in DNS traffic (Detect C2 channels that evade signature-based detection)  
**M-ATH approach:** Inter-query interval statistics (mean, std, CV) to separate periodic C2 callbacks from normal DNS traffic.

## How to use
1. Put DNS CSV files into `input/` (e.g. SentinelOne PowerQuery results with event.dns.request, timestamp, endpoint identifier)
2. Run all cells
3. Review flagged flows in `output/`

In [ ]:
pass  # Placeholder (removed environment-specific output)

## PREPARE — Plan your Approach

- **Select Topic:** DNS-layer C2 beaconing — adversaries use periodic DNS queries to maintain command-and-control channels that evade signature-based detection (MITRE ATT&CK [T1071.004](https://attack.mitre.org/techniques/T1071/004/) Application Layer Protocol: DNS).
- **Research Topic:** Inter-query interval statistics, coefficient of variation for periodicity detection, DNS beaconing literature.
- **Identify Datasets:** DNS logs with timestamps, query domains, and endpoint identifiers from EDR/SIEM.
- **Select Algorithms:** CV-based periodicity detection — compute mean/std/CV of inter-query intervals per endpoint-domain pair; flag flows with low CV (high regularity).

In [ ]:
# Scenario mode: run from scenarios/beaconing_behavior_in_dns_traffic_detect_c2_channels_that_evade_signature_based_detection/
import os
import sys
from pathlib import Path
SCENARIO_DIR = Path('.').resolve()
REPO_ROOT = SCENARIO_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True

In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 120)

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals() and hasattr(p, 'is_relative_to') and p.is_relative_to(REPO_ROOT):
            return p.relative_to(REPO_ROOT)
        return p
    except (ValueError, AttributeError):
        return p

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')

## EXECUTE — Experimentation Time

- **Gather Data:** Load DNS CSVs from `input/`, detect domain/time/endpoint columns.
- **Pre-Process Data:** Normalize columns, sort by timestamp, compute inter-query intervals per endpoint-domain pair.
- **Apply:** Calculate CV and periodicity metrics for each flow group.
- **Analyze:** Flag flows with low CV (regular intervals) above minimum query thresholds.
- **Escalate Critical Findings:** Endpoint-domain pairs with very low CV and many queries warrant immediate C2 investigation.

## Load DNS data

In [ ]:
csv_paths = sorted(glob.glob(str(INPUT_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_paths)} CSV file(s).')

if not csv_paths:
    raise FileNotFoundError(f'No CSV files in {_rel(INPUT_DIR)}. Add DNS logs (event.dns.request, timestamp, endpoint id).')

dfs = []
for p in csv_paths:
    df = pd.read_csv(p)
    try:
        src_rel = str(Path(p).relative_to(REPO_ROOT)) if 'REPO_ROOT' in globals() and Path(p).is_relative_to(REPO_ROOT) else p
    except (ValueError, AttributeError):
        src_rel = p
    df['_source_file'] = src_rel
    dfs.append(df)
raw = pd.concat(dfs, ignore_index=True)

# Normalize column names
domain_col = next((c for c in raw.columns if 'dns.request' in c.lower() or c == 'query' or c == 'domain'), None)
time_col = next((c for c in raw.columns if c in ('_time', 'event.time', 'timestamp', '@timestamp', 'time') or 'time' in c.lower()), None)
endpoint_col = next((c for c in raw.columns if 'agent.uuid' in c or 'endpoint.name' in c or 'host' in c.lower() or 'endpoint' in c.lower()), None)

if not domain_col:
    raise ValueError(f'No DNS request column found. Columns: {list(raw.columns)}')
if not endpoint_col:
    endpoint_col = 'agent.uuid' if 'agent.uuid' in raw.columns else 'endpoint.name'
if endpoint_col not in raw.columns:
    raw['_endpoint'] = raw.index.astype(str)
    endpoint_col = '_endpoint'

raw = raw.rename(columns={domain_col: 'domain', endpoint_col: 'endpoint'})
if time_col and time_col in raw.columns:
    raw['_time'] = pd.to_datetime(raw[time_col], errors='coerce', utc=True)
else:
    raw['_time'] = pd.to_datetime('now', utc=True) + pd.to_timedelta(raw.index, unit='s')
    print('Warning: no timestamp column. Using synthetic timestamps.')

raw = raw[['domain', 'endpoint', '_time', '_source_file']].dropna(subset=['domain', 'endpoint'])
raw['domain'] = raw['domain'].astype(str).str.strip().str.lower().str.rstrip('.')
raw = raw[raw['domain'].str.len() > 0]
print(f'Loaded {len(raw):,} DNS records.')

## Compute inter-query intervals and periodicity

In [ ]:
def compute_beaconing_metrics(group: pd.DataFrame) -> dict:
    """Compute interval statistics for (endpoint, domain) flow."""
    group = group.sort_values('_time').drop_duplicates(subset=['_time'])
    if len(group) < 3:
        return None
    ts = group['_time'].astype('int64') // 1_000_000_000  # seconds
    intervals = np.diff(ts.values)
    intervals = intervals[intervals > 0]  # exclude duplicates
    if len(intervals) < 2:
        return None
    mean_i = np.mean(intervals)
    std_i = np.std(intervals)
    cv = std_i / mean_i if mean_i > 0 else float('inf')  # coefficient of variation
    # Low CV suggests regular/periodic beaconing
    return {
        'query_count': len(group),
        'interval_mean_s': mean_i,
        'interval_std_s': std_i,
        'interval_cv': cv,
        'interval_min_s': np.min(intervals),
        'interval_max_s': np.max(intervals),
    }

flows = raw.groupby(['endpoint', 'domain'], as_index=False)
results = []
for (endpoint, domain), grp in flows:
    m = compute_beaconing_metrics(grp)
    if m:
        results.append({
            'endpoint': endpoint,
            'domain': domain,
            **m
        })

flows_df = pd.DataFrame(results)
print(f'Computed metrics for {len(flows_df):,} (endpoint, domain) flows.')

## Flag suspicious beaconing

In [ ]:
# Beaconing: low CV (regular intervals), enough queries, reasonable mean interval (e.g. 30s–1h)
MIN_QUERIES = 5
MAX_CV = 0.5  # coefficient of variation threshold
MIN_MEAN_S = 10
MAX_MEAN_S = 7200  # 2 hours

flagged = flows_df[
    (flows_df['query_count'] >= MIN_QUERIES) &
    (flows_df['interval_cv'] <= MAX_CV) &
    (flows_df['interval_mean_s'] >= MIN_MEAN_S) &
    (flows_df['interval_mean_s'] <= MAX_MEAN_S)
].copy()
flagged = flagged.sort_values('interval_cv').reset_index(drop=True)

print(f'Flagged {len(flagged):,} flows as potential beaconing.')
if len(flagged) > 0:
    display(flagged.head(20))

## ACT — Wrapping Up the Investigation

- **Document Findings:** Flagged beaconing flows with CV metrics exported for analyst review and ticketing.
- **Preserve Hunt:** Results archived to `output/beaconing_dns_results.csv`.
- **Create Detections / Playbooks:** Confirmed beaconing domains can feed DNS blocklists or scheduled detection rules.

In [ ]:
out_path = OUTPUT_DIR / 'beaconing_dns_results.csv'
flagged.to_csv(out_path, index=False)
print(f'Saved to {_rel(out_path)}')

## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New beaconing patterns, jitter techniques, or DNS tunneling variants discovered during analysis become future hunt hypotheses.
- **Communicate Findings:** Share confirmed DNS C2 channels with SOC, DNS administrators, and threat intelligence teams.
- **Feed Improvements Back:** Tune CV thresholds, minimum query counts, and interval bounds based on true/false positive rates; add confirmed benign periodic domains to exclusions.
- **Measure Effectiveness:** Track detection counts and confirmed C2 channels across successive runs to assess hunting maturity.